🔹 Step 1: Environment Setup

In [272]:
# Install required libraries (run once)
# pip install pandas numpy matplotlib scikit-learn tensorflow

In [273]:
!python --version

Python 3.12.12


🔹 Step 2: Import Libraries

In [274]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Input, Bidirectional
from sklearn.utils.class_weight import compute_class_weight
from tensorflow.keras.callbacks import EarlyStopping

🔹 Step 3: Load & Inspect Dataset

In [275]:
df = pd.read_csv("Job_3_Resource_sentiment.csv")
print(df.columns)

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')


In [276]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


In [277]:
df.columns

Index(['2401', 'Borderlands', 'Positive',
       'im getting on borderlands and i will murder you all ,'],
      dtype='object')

In [278]:
df.rename(columns={'Positive': 'sentiment'}, inplace=True)
df.rename(columns={'im getting on borderlands and i will murder you all ,': 'text'}, inplace=True)

In [279]:
df.columns

Index(['2401', 'Borderlands', 'sentiment', 'text'], dtype='object')

In [280]:
df.sample(5)

,2401,Borderlands,sentiment,text
68641,3754,Cyberpunk2077,Negative,Fuckin noice
49926,6171,FIFA,Irrelevant,@ trentaa98 I loved it when you lost to Diogo ...
16828,9684,PlayStation5(PS5),Neutral,niggas be worried over the cost of the PS5 & a...
55941,11202,TomClancysRainbowSix,Negative,Microsoft @Rainbow6Game Issue with matchmaking...
50914,6338,FIFA,Negative,@chaplinez70 morning. You might not agree with...


In [281]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   2401         74681 non-null  int64 
 1   Borderlands  74681 non-null  object
 2   sentiment    74681 non-null  object
 3   text         73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


In [282]:
df = df[['text', 'sentiment']]

In [283]:
df.sample(5)

,text,sentiment
45773,honestly stupid how fast this is,Negative
23034,Help me win this fantastic CS: GO raffle from ...,Positive
10504,a't wait!,Positive
62059,I was inspired by the gold of @ DemolitionRanc...,Positive
23245,@ Pinkwardlol is so mad you lost that CSGO gam...,Irrelevant


In [284]:
df

,text,sentiment
0,I am coming to the borders and I will kill you...,Positive
1,im getting on borderlands and i will kill you ...,Positive
2,im coming on borderlands and i will murder you...,Positive
3,im getting on borderlands 2 and i will murder ...,Positive
4,im getting into borderlands and i can murder y...,Positive
...,...,...
74676,Just realized that the Windows partition of my...,Positive
74677,Just realized that my Mac window partition is ...,Positive
74678,Just realized the windows partition of my Mac ...,Positive
74679,Just realized between the windows partition of...,Positive


In [285]:
df.shape

(74681, 2)

In [286]:
df.isnull().sum()

,0
text,686
sentiment,0


In [287]:
print(df['sentiment'].value_counts())

sentiment
Negative      22542
Positive      20831
Neutral       18318
Irrelevant    12990
Name: count, dtype: int64


In [288]:
df.duplicated().sum()

np.int64(4909)

🔹 Step 4: Data Cleaning

In [289]:
df.dropna(inplace=True)

In [290]:
df.isnull().sum()

,0
text,0
sentiment,0


In [291]:
df['text'] = df['text'].astype(str)

In [292]:
df['text']

,text
0,I am coming to the borders and I will kill you...
1,im getting on borderlands and i will kill you ...
2,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...
4,im getting into borderlands and i can murder y...
...,...
74676,Just realized that the Windows partition of my...
74677,Just realized that my Mac window partition is ...
74678,Just realized the windows partition of my Mac ...
74679,Just realized between the windows partition of...


In [293]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    return text.strip()

df['text'] = df['text'].apply(clean_text)

In [294]:
df['text']

,text
0,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...
3,im getting on borderlands and i will murder y...
4,im getting into borderlands and i can murder y...
...,...
74676,just realized that the windows partition of my...
74677,just realized that my mac window partition is ...
74678,just realized the windows partition of my mac ...
74679,just realized between the windows partition of...


🔹 Step 5: Encode Target Labels

In [295]:
encoder = LabelEncoder()
df['sentiment_encoded'] = encoder.fit_transform(df['sentiment'])

print("Label mapping:")
for i, c in enumerate(encoder.classes_):
    print(i, "->", c)

Label mapping:
0 -> Irrelevant
1 -> Negative
2 -> Neutral
3 -> Positive


In [296]:
encoder.classes_

array(['Irrelevant', 'Negative', 'Neutral', 'Positive'], dtype=object)

In [297]:
df.sample(10)

,text,sentiment,sentiment_encoded
64787,touch down your team to victory with incredibl...,Neutral,2
55436,best home,Negative,1
45160,brilliant innovation get the scoop when verizo...,Neutral,2
38917,king shit,Negative,1
56098,rainbowgameunk to play ranked a pc is difficul...,Negative,1
24965,yeah heres more proof that your bossbabe mlm p...,Irrelevant,0
15125,dota after months lets faking goo,Positive,3
71771,ghostrecon all the backpacks are not fitting p...,Negative,1
54723,the love i seeing of all the new tweets keep e...,Irrelevant,0
29335,all right lets do it the road to gold is alre...,Neutral,2


🔹 Step 6: Text Tokenization & Padding

In [298]:
max_words = 20000
max_length = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['text'])

X = pad_sequences(
    tokenizer.texts_to_sequences(df['text']),
    maxlen=max_length,
    padding='post'
)

y = df['sentiment_encoded'].values

In [299]:
tokenizer

In [300]:
df

,text,sentiment,sentiment_encoded
0,i am coming to the borders and i will kill you...,Positive,3
1,im getting on borderlands and i will kill you all,Positive,3
2,im coming on borderlands and i will murder you...,Positive,3
3,im getting on borderlands and i will murder y...,Positive,3
4,im getting into borderlands and i can murder y...,Positive,3
...,...,...,...
74676,just realized that the windows partition of my...,Positive,3
74677,just realized that my mac window partition is ...,Positive,3
74678,just realized the windows partition of my mac ...,Positive,3
74679,just realized between the windows partition of...,Positive,3


In [301]:
X.shape

(73995, 100)

In [302]:
X.dtype

dtype('int32')

In [303]:
y.shape

(73995,)

In [304]:
y.dtype

dtype('int64')

In [305]:
X.ndim

2

In [306]:
y.ndim

1

In [307]:
X.nbytes

29598000

In [308]:
y.nbytes

591960

In [309]:
X

array([[   3,  101,  377, ...,    0,    0,    0],
       [  31,  158,   14, ...,    0,    0,    0],
       [  31,  377,   14, ...,    0,    0,    0],
       ...,
       [  22, 1837,    2, ...,    0,    0,    0],
       [  22, 1837,  693, ...,    0,    0,    0],
       [  22,   32,    2, ...,    0,    0,    0]], dtype=int32)

In [310]:
y

array([3, 3, 3, ..., 3, 3, 3])

🔹 Step 7: Train / Validation / Test Split (MANDATORY)

In [311]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42
)

print("\nTrain:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)



Train: (51796, 100)
Validation: (11099, 100)
Test: (11100, 100)


Handle Bias (Class Weight)

In [312]:
print(np.bincount(y_train))

[ 9012 15650 12676 14458]


In [313]:
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.unique(y_train),
    y=y_train
)

class_weights_dict = dict(enumerate(class_weights))
print("\nClass Weights:", class_weights_dict)


Class Weights: {0: np.float64(1.4368619618286729), 1: np.float64(0.8274121405750798), 2: np.float64(1.0215367623856106), 3: np.float64(0.8956287176649605)}


In [314]:
print(np.bincount(y_train))

[ 9012 15650 12676 14458]
